# Multitaper Spectral Analysis — Reference Notebook

**Neural Data Science (WP7)** · LMU Biology

---

This notebook is a self-contained reference for spectral estimation
methods applied to hippocampal local field potentials (LFP).  It covers:

1. **Periodogram** — the simplest spectral estimator and its limitations
2. **Spectral leakage** — how window length affects frequency resolution
3. **Welch's method** — variance reduction via segment averaging
4. **Multitaper estimation** — DPSS tapers and the NW trade-off
5. **Filtering and envelopes** — Butterworth bandpass + Hilbert transform
6. **Depth-wise spectra** — laminar variation of theta and gamma power

All code runs on `numpy`, `scipy`, `matplotlib`, and the course helper
library `wp7_helpers` (which optionally uses `ghostipy` for faster
multitaper computation).

In [ ]:
import sys
import os

import numpy as np
import matplotlib.pyplot as plt
from scipy import io, signal

sys.path.insert(0, '../lib')
from wp7_helpers import psd_multitaper

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 4)

In [ ]:
# Load 16-channel hippocampal LFP (single shank, 1250 Hz)
data_path = '../data/spectral/ws_data_1shank.mat'
if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"Data not found at {data_path!r}. "
        "Adjust data_path to point to ws_data_1shank.mat"
    )

mat = io.loadmat(data_path)
lfps = mat['lfps'].astype(np.float64)
fs = 1250.0  # Hz

n_samples, n_channels = lfps.shape
print(f'LFP: {lfps.shape}  ({n_samples / fs:.1f} s, {n_channels} channels, {fs:.0f} Hz)')

## 1. Raw LFP and the periodogram

The **periodogram** is the squared magnitude of the discrete Fourier
transform, normalised to give power spectral density:

$$
\hat{S}(f_k) = \frac{1}{N}\,\bigl|\textstyle\sum_{n=0}^{N-1} x_n\,e^{-j2\pi f_k n/N}\bigr|^2
$$

It is an **unbiased** but **inconsistent** estimator — its variance does
not decrease as $N$ grows.  In practice the periodogram is too noisy to
be useful by itself; the methods below all aim to reduce that variance.

In [ ]:
# Select channel 6 (strong theta) and use 10 s of data
ch = 6
duration_sec = 10.0
n_seg = int(duration_sec * fs)
x = lfps[:n_seg, ch]
t = np.arange(n_seg) / fs

# Raw trace
fig, ax = plt.subplots(figsize=(12, 2.5))
ax.plot(t, x, linewidth=0.4)
ax.set(xlabel='Time (s)', ylabel='Amplitude (a.u.)',
       title=f'Raw LFP — channel {ch}, first {duration_sec:.0f} s')
plt.tight_layout()
plt.show()

# Periodogram
f_per, psd_per = signal.periodogram(x, fs=fs, window='boxcar',
                                     detrend='constant',
                                     return_onesided=True)

fig, ax = plt.subplots()
mask = (f_per >= 1) & (f_per <= 150)
ax.semilogy(f_per[mask], psd_per[mask], linewidth=0.5)
ax.set(xlabel='Frequency (Hz)', ylabel='PSD (a.u.²/Hz)',
       title='Direct periodogram (10 s, boxcar window)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Spectral leakage and window length

The periodogram implicitly applies a **rectangular (boxcar) window** to
the finite data segment.  The Fourier transform of that window — with
its broad side-lobes — leaks power from strong spectral peaks into
neighbouring frequencies.

**Shorter windows** broaden the main lobe, reducing frequency resolution
and increasing leakage.  Conversely, **longer windows** sharpen the main
lobe but cannot average out noise from a single realisation.

In [ ]:
# Periodogram for several window lengths
win_lengths = [512, 1024, 2048, 4096, n_seg]

fig, axes = plt.subplots(len(win_lengths), 1, figsize=(10, 8), sharex=True)
for ax, N in zip(axes, win_lengths):
    f_w, p_w = signal.periodogram(x[:N], fs=fs, window='boxcar',
                                   detrend='constant',
                                   return_onesided=True)
    sel = (f_w >= 1) & (f_w <= 120)
    ax.semilogy(f_w[sel], p_w[sel], linewidth=0.6)
    ax.set_ylabel(f'N={N}')
    ax.grid(True, alpha=0.2)

axes[-1].set_xlabel('Frequency (Hz)')
fig.suptitle('Periodogram vs window length — spectral leakage', y=1.01)
fig.tight_layout()
plt.show()

## 3. Welch's method

Welch's estimator splits the signal into overlapping, windowed segments,
computes a periodogram for each, and averages them.  This reduces
variance at the cost of frequency resolution (set by `nperseg`).

Key parameters:
- **`nperseg`** — segment length in samples; determines frequency resolution
  ($\Delta f = f_s / \texttt{nperseg}$).
- **overlap** — fraction of overlap between consecutive segments; 50–75 %
  is typical.
- **zero-padding** (`nfft > nperseg`) — interpolates the frequency axis
  without improving true resolution.

In [ ]:
# Welch PSD: compare segment lengths at 50 % overlap
nperseg_list = [256, 512, 1024, 2048]
x_long = lfps[:, ch]  # full recording

fig, ax = plt.subplots(figsize=(10, 5))
for nperseg in nperseg_list:
    f_w, p_w = signal.welch(x_long, fs=fs, nperseg=nperseg,
                             noverlap=nperseg // 2, nfft=2 * nperseg)
    sel = (f_w >= 1) & (f_w <= 120)
    ax.semilogy(f_w[sel], p_w[sel], linewidth=0.9,
                label=f'nperseg={nperseg} ({nperseg/fs:.2f} s)')

ax.axvspan(6, 10, alpha=0.10, color='red', label='Theta')
ax.axvspan(30, 90, alpha=0.07, color='blue', label='Gamma')
ax.set(xlabel='Frequency (Hz)', ylabel='PSD (a.u.²/Hz)',
       title='Welch PSD — effect of segment length')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Multitaper estimation

### DPSS tapers and the time–halfbandwidth product (NW)

Instead of splitting the signal into segments, multitaper estimation
applies **multiple orthogonal tapers** (Discrete Prolate Spheroidal
Sequences, DPSS) to the *same* data block and averages the resulting
periodograms.  The parameter **NW** (time–halfbandwidth product)
controls the trade-off:

| NW | Tapers $K = 2\text{NW} - 1$ | Resolution bandwidth | Variance |
|----|----|----|----|  
| 2  | 3  | narrow (sharp peaks) | higher |
| 3  | 5  | moderate | moderate |
| 4  | 7  | wide (smoother) | lower |

The frequency resolution is approximately $2\,\text{NW} \cdot f_s / N$ Hz.

Multitaper estimation is the **gold standard** for neural spectral
analysis because it provides optimal bias–variance control without
requiring segment averaging (and thus without sacrificing frequency
resolution from short windows).

In [ ]:
# Multitaper PSD with wp7_helpers: NW sweep
nw_values = [2.0, 3.0, 4.0]

fig, ax = plt.subplots(figsize=(10, 5))

# Welch reference
f_ref, p_ref = signal.welch(x_long, fs=fs, nperseg=int(2 * fs))
sel = (f_ref >= 1) & (f_ref <= 150)
ax.semilogy(f_ref[sel], p_ref[sel], 'k--', alpha=0.4, linewidth=0.8,
            label='Welch (2 s window)')

for nw in nw_values:
    freqs, psd = psd_multitaper(x_long, fs=fs, nw=nw)
    sel = (freqs >= 1) & (freqs <= 150)
    k = int(2 * nw - 1)
    ax.semilogy(freqs[sel], psd[sel], linewidth=0.9,
                label=f'NW={nw:.0f} ({k} tapers)')

ax.axvspan(6, 10, alpha=0.10, color='red')
ax.axvspan(30, 90, alpha=0.07, color='blue')
ax.set(xlabel='Frequency (Hz)', ylabel='PSD (a.u.²/Hz)',
       title='Multitaper PSD — NW sweep (full recording)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Method comparison: 1 s vs 10 s segments

On short segments (1 s), the periodogram is extremely noisy; Welch can
barely average 2–4 sub-windows; but the multitaper still provides a
smooth estimate because it averages over *tapers* rather than *time
segments*.

On longer segments (10 s), all methods improve, but the multitaper
retains the best bias–variance trade-off.

In [ ]:
segments = {'1 s': lfps[:int(1 * fs), ch],
            '10 s': lfps[:int(10 * fs), ch]}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (label, seg) in zip(axes, segments.items()):
    N = len(seg)

    # Periodogram
    f_p, p_p = signal.periodogram(seg, fs=fs, window='boxcar', detrend='constant')
    sel = (f_p >= 1) & (f_p <= 120)
    ax.semilogy(f_p[sel], p_p[sel], alpha=0.4, linewidth=0.5, label='Periodogram')

    # Welch
    nperseg_w = max(N // 4, 64)
    f_w, p_w = signal.welch(seg, fs=fs, nperseg=nperseg_w, noverlap=nperseg_w // 2)
    sel_w = (f_w >= 1) & (f_w <= 120)
    ax.semilogy(f_w[sel_w], p_w[sel_w], linewidth=1.0, label=f'Welch (nperseg={nperseg_w})')

    # Multitaper
    f_mt, p_mt = psd_multitaper(seg, fs=fs, nw=3.0)
    sel_mt = (f_mt >= 1) & (f_mt <= 120)
    ax.semilogy(f_mt[sel_mt], p_mt[sel_mt], linewidth=1.0, label='Multitaper (NW=3)')

    ax.set(xlabel='Frequency (Hz)', title=f'{label} segment')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('PSD (a.u.²/Hz)')
fig.suptitle('Periodogram vs Welch vs Multitaper', y=1.02)
fig.tight_layout()
plt.show()

## 5. Filtering and envelope extraction

Band-pass filtering isolates an oscillatory component; the **Hilbert
transform** of the filtered signal gives the analytic signal, whose
magnitude is the **instantaneous amplitude envelope** and whose angle
is the **instantaneous phase**.

This is the basis for all phase-based analyses (phase-amplitude
coupling, spike–phase locking, etc.).

In [ ]:
# Band-pass filtering with scipy
ch0 = lfps[:int(16 * fs), 0]  # 16 s segment, channel 0
t_filt = np.arange(len(ch0)) / fs

# Theta (6–10 Hz) and gamma (40–90 Hz)
b_th, a_th = signal.butter(4, [6.0, 10.0], btype='bandpass', fs=fs)
b_gm, a_gm = signal.butter(4, [40.0, 90.0], btype='bandpass', fs=fs)

theta = signal.filtfilt(b_th, a_th, ch0)
gamma = signal.filtfilt(b_gm, a_gm, ch0)

# Hilbert envelopes
theta_env = np.abs(signal.hilbert(theta))
gamma_env = np.abs(signal.hilbert(gamma))

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

axes[0].plot(t_filt, ch0, linewidth=0.3, color='gray')
axes[0].set_ylabel('Raw LFP')
axes[0].set_title('Channel 0 — raw LFP and band-limited components')

axes[1].plot(t_filt, theta, linewidth=0.5, label='Theta (6–10 Hz)')
axes[1].plot(t_filt, theta_env, color='black', alpha=0.7, linewidth=1.0, label='Envelope')
axes[1].legend(fontsize=8)
axes[1].set_ylabel('Amplitude')

axes[2].plot(t_filt, gamma, linewidth=0.5, label='Gamma (40–90 Hz)')
axes[2].plot(t_filt, gamma_env, color='black', alpha=0.7, linewidth=1.0, label='Envelope')
axes[2].legend(fontsize=8)
axes[2].set(xlabel='Time (s)', ylabel='Amplitude')

fig.tight_layout()
plt.show()

## 6. Depth-wise spectral profiles

The 16-channel linear probe spans several hippocampal layers.  Different
layers have distinct oscillatory generators:

- **Theta power** (6–10 Hz) peaks near stratum lacunosum-moleculare
  and reverses phase across the pyramidal layer.
- **Gamma power** (30–90 Hz) has compartment-specific generators: slow
  gamma in stratum radiatum, fast gamma deeper.

Computing the PSD channel-by-channel reveals this laminar structure.

In [ ]:
# Multitaper PSD per channel (10 s segment)
depth_sec = 10.0
n_depth = int(depth_sec * fs)

psd_depth = []
for ch_i in range(n_channels):
    f_d, p_d = psd_multitaper(lfps[:n_depth, ch_i], fs=fs, nw=3.0)
    psd_depth.append(p_d)

psd_depth = np.array(psd_depth)  # (n_channels, n_freqs)
freq_mask = (f_d >= 1) & (f_d <= 120)
psd_db = 10 * np.log10(psd_depth[:, freq_mask] + 1e-20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].pcolormesh(f_d[freq_mask], np.arange(n_channels),
                         psd_db, shading='auto', cmap='viridis')
axes[0].set(xlabel='Frequency (Hz)', ylabel='Channel (superficial → deep)',
            title='Depth-wise multitaper PSD (NW=3, 10 s)')
plt.colorbar(im, ax=axes[0], label='Power (dB)')

# Line plot: theta vs gamma power per channel
theta_mask = (f_d >= 6) & (f_d <= 10)
gamma_mask = (f_d >= 30) & (f_d <= 90)
theta_power = 10 * np.log10(psd_depth[:, theta_mask].mean(axis=1) + 1e-20)
gamma_power = 10 * np.log10(psd_depth[:, gamma_mask].mean(axis=1) + 1e-20)

axes[1].plot(theta_power, np.arange(n_channels), 'o-', label='Theta (6–10 Hz)')
axes[1].plot(gamma_power, np.arange(n_channels), 's-', label='Gamma (30–90 Hz)')
axes[1].set(xlabel='Mean band power (dB)', ylabel='Channel',
            title='Band power vs depth')
axes[1].legend()
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Summary

| Method | Variance reduction | Frequency resolution | Key parameter |
|--------|-------------------|---------------------|---------------|
| Periodogram | None | Best (for given N) | Window length |
| Welch | Segment averaging | Set by `nperseg` | `nperseg`, overlap |
| Multitaper | Taper averaging | Set by NW and N | NW |

**Practical guidance:**
- Use **Welch** when you have a long, stationary recording and want a
  quick, low-variance estimate.
- Use **multitaper** (NW = 2–4) when you need the best resolution for a
  given data length — this is the standard for neural oscillation analysis.
- **Always detrend** before computing any PSD.
- Plot on a **log scale** to see the 1/f background and oscillatory peaks.
- The `wp7_helpers.psd_multitaper` function handles DPSS computation and
  taper averaging for you; see `lib/wp7_helpers.py` for details.